[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Classic_Control_Introduction.ipynb)

# Classic Control: Control theory problems from the classic RL literature

<br><br>

**What is an environment?** In reinforcement learning, an *environment* is the
task the agent has to solve. At every moment the environment is in some
**state** (a description of the situation). The agent picks an **action**, and
the environment replies with a **reward** (a number saying how good that was)
and the next state. The agent's job is to choose actions that collect as much
reward as possible.

In this notebook we look at four famous **classic-control** environments. Each
one is a small physics task &mdash; balancing a pole, swinging a bar, driving a
car up a hill, holding a pendulum upright. They are favourites in RL research
because they are simple to state but not trivial to solve.

Unlike the maze, these environments have **continuous state spaces** (the state
is made of real numbers, so there are infinitely many possible states).
Plain table-based methods cannot store a value for every state, so we will need
two extra tools later in the course:

- Extend tabular methods using **discretization** and **tile coding**.
- Use **function approximators** (such as neural networks).

For now we just want to *meet* each environment: see what its state looks like,
what actions it allows, and watch a random agent try it out.

In [1]:
# @title Setup code (not important) - Run this cell by pressing "Shift + Enter"


!pip install -qq gym==0.23.0


import matplotlib
from matplotlib import animation
from IPython.display import HTML


def display_video(frames):
    # Copied from: https://colab.research.google.com/github/deepmind/dm_control/blob/master/tutorial.ipynb
    orig_backend = matplotlib.get_backend()
    matplotlib.use('Agg')
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    matplotlib.use(orig_backend)
    ax.set_axis_off()
    ax.set_aspect('equal')
    ax.set_position([0, 0, 1, 1])
    im = ax.imshow(frames[0])
    def update(frame):
        im.set_data(frame)
        return [im]
    anim = animation.FuncAnimation(fig=fig, func=update, frames=frames,
                                    interval=50, blit=True, repeat=False)
    return HTML(anim.to_html5_video())


def test_env(environment, episodes=10):
    frames = []
    for episode in range(episodes):
        state = environment.reset()
        done = False
        frames.append(environment.render(mode="rgb_array"))

        while not done:
            action = environment.action_space.sample()
            next_state, reward, done, extra_info = environment.step(action)
            img = environment.render(mode="rgb_array")
            frames.append(img)
            state = next_state

    return display_video(frames)



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.4/624.4 kB 8.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# Gym provides the classic-control environments we will explore.
import gym
# NumPy for numerical work.
import numpy as np
# 'display' lets us refresh output inside the notebook.
from IPython import display
# Matplotlib for drawing the environment pictures.
from matplotlib import pyplot as plt
# Show plots right here in the notebook.
%matplotlib inline

## CartPole: Keep the tip of the pole straight.

A pole is hinged on top of a cart. The agent pushes the cart left or right to
keep the pole from falling over. Each time step the pole stays up earns a
reward, so the goal is to balance it for as long as possible.

In [3]:
# Create the CartPole environment by its Gym name.
env = gym.make('CartPole-v1')
# Run 1 episode with random actions and show it as a video (test_env is in the setup cell).
test_env(env, 1)
# Release the environment's resources.
env.close()

##### The state

The states of the cartpole task will be represented by a vector of four real numbers:

        Num     Observation               Min                     Max
        0       Cart Position             -4.8                    4.8
        1       Cart Velocity             -Inf                    Inf
        2       Pole Angle                -0.418 rad (-24 deg)    0.418 rad (24 deg)
        3       Pole Angular Velocity     -Inf                    Inf


In [4]:
# Show the observation (state) space: the four numbers describing the cart and pole.
env.observation_space

Box([-4.8000002e+00 -3.4028235e+38 -4.1887903e-01 -3.4028235e+38], [4.8000002e+00 3.4028235e+38 4.1887903e-01 3.4028235e+38], (4,), float32)

##### The actions available

We can perform two actions in this environment:

        0     Push cart to the left.
        1     Push cart to the right.



In [5]:
# Show the action space: the moves the agent can make (push left or right).
env.action_space

Discrete(2)

## Acrobot: Swing the bar up to a certain height.

Acrobot is a two-link arm hanging down, hinged at the top. The agent can only
apply torque at the middle joint. By swinging back and forth it must build up
enough momentum to lift the free end above a target height.

In [6]:
# Create the Acrobot environment.
env = gym.make('Acrobot-v1')
# Run 1 episode with random actions and show it as a video.
test_env(env, 1)
# Release the environment's resources.
env.close()

##### The state

The states of the cartpole task will be represented by a vector of six real numbers. The first two are the cosine and sine of the first joint. The next two are the cosine and sine of the other joint. The last two are the angular velocities of each joint.
    
$\cos(\theta_1), \sin(\theta_1), \cos(\theta_2), \sin(\theta_2), \dot\theta_1, \dot\theta_2$

In [7]:
# Show the observation space: the six numbers describing the two joints.
env.observation_space

Box([ -1.        -1.        -1.        -1.       -12.566371 -28.274334], [ 1.        1.        1.        1.       12.566371 28.274334], (6,), float32)

##### The actions available

We can perform two actions in this environment:

    0    Apply +1 torque on the joint between the links.
    1    Apply -1 torque on the joint between the links.

In [8]:
# Show the action space: apply +1 or -1 torque on the joint.
env.action_space

Discrete(3)

## MountainCar: Reach the goal from the bottom of the valley.

An underpowered car sits in a valley. Its engine is too weak to drive straight
up the hill, so the agent must rock the car back and forth to build momentum
and reach the flag at the top.

In [9]:
# Create the MountainCar environment.
env = gym.make('MountainCar-v0')
# Run 1 episode with random actions and show it as a video.
test_env(env, 1)
# Release the environment's resources.
env.close()

##### The state

The observation space consists of the car position $\in [-1.2, 0.6]$ and car velocity $\in [-0.07, 0.07]$

In [10]:
# Show the observation space: the car's position and velocity.
env.observation_space

Box([-1.2  -0.07], [0.6  0.07], (2,), float32)

##### The actions available


The actions available three:

    0    Accelerate to the left.
    1    Don't accelerate.
    2    Accelerate to the right.

In [11]:
# Show the action space: accelerate left, do nothing, or accelerate right.
env.action_space

Discrete(3)

## Pendulum: swing it and keep it upright

A pendulum starts in a random position. The agent applies torque to swing it up
and then hold it balanced pointing straight up. Unlike the tasks above, the
action here is a **continuous** value (any torque in a range), not a choice from
a short list.

In [12]:
# Create the Pendulum environment.
env = gym.make('Pendulum-v1')
# Run 1 episode with random actions and show it as a video.
test_env(env, 1)
# Release the environment's resources.
env.close()

##### The state

The state is represented by a vector of three values representing $\cos(\theta), \sin(\theta)$ and speed ($\theta$ is the angle of the pendulum).

In [13]:
# Show the observation space: cos(theta), sin(theta) and the pendulum's speed.
env.observation_space

Box([-1. -1. -8.], [1. 1. 8.], (3,), float32)

##### The actions available

The action is a real number in the interval $[-2, 2]$ that represents the torque applied on the pendulum.

In [14]:
# Show the action space: a single real-valued torque in the range [-2, 2].
env.action_space

Box(-2.0, 2.0, (1,), float32)

## Resources

[[1] OpenAI gym: classic control environments](https://gym.openai.com/envs/#classic_control)